# Google OAuth in Backstage

Diese Anleitung zeigt die **einfachste und kürzeste Integration von Google OAuth in Backstage**.

**Rahmenbedingungen**

- Lokale Backstage-Installation
- Keine Kubernetes-Konfiguration
- Google dient als Login-Provider
- Backstage läuft lokal auf:
  - Frontend: `http://localhost:3000`
  - Backend: `http://localhost:7007`


## 1. OAuth-Client bei Google erstellen

In der **Google Cloud Console**:

1. Ein Projekt erstellen oder auswählen.
2. Unter **Google Auth Platform → Branding** den OAuth-Zustimmungsbildschirm konfigurieren.
3. Bei einer externen Anwendung den eigenen Google-Account als Testnutzer eintragen.
4. Unter **Clients** einen neuen Client vom Typ **Web application** erstellen.
5. Die folgenden Werte eintragen:

```text
Authorized JavaScript origin:
http://localhost:3000

Authorized redirect URI:
http://localhost:7007/api/auth/google/handler/frame
```

Die Redirect-URI muss exakt mit der in Google hinterlegten URI übereinstimmen.

Nach dem Erstellen des OAuth-Clients werden folgende Werte benötigt:

```text
Client ID
Client Secret
```


## 2. Google-Provider installieren

Der Google-Provider wird im Backend-Paket der Backstage-Installation installiert.

Im Root-Verzeichnis von Backstage ausführen:


In [ ]:
# Im Root-Verzeichnis der Backstage-Installation ausführen
yarn --cwd packages/backend add @backstage/plugin-auth-backend-module-google-provider

## 3. Backend-Modul aktivieren

In der Datei:

```text
packages/backend/src/index.ts
```

müssen das Auth-Backend und das Google-Provider-Modul registriert sein:


In [ ]:
backend.add(import('@backstage/plugin-auth-backend'));

backend.add(
  import('@backstage/plugin-auth-backend-module-google-provider'),
);

Falls `@backstage/plugin-auth-backend` bereits vorhanden ist, muss nur das Google-Modul ergänzt werden.


## 4. Google-Provider konfigurieren

Die lokale OAuth-Konfiguration wird in `app-config.local.yaml` eingetragen:


In [ ]:
auth:
  environment: development
  providers:
    google:
      development:
        clientId: ${AUTH_GOOGLE_CLIENT_ID}
        clientSecret: ${AUTH_GOOGLE_CLIENT_SECRET}
        signIn:
          resolvers:
            - resolver: emailMatchingUserEntityProfileEmail

Die beiden Zugangsdaten werden als Umgebungsvariablen gesetzt:


In [ ]:
export AUTH_GOOGLE_CLIENT_ID='deine-client-id.apps.googleusercontent.com'
export AUTH_GOOGLE_CLIENT_SECRET='dein-client-secret'

Der Resolver `emailMatchingUserEntityProfileEmail` vergleicht die E-Mail-Adresse des Google-Accounts mit der E-Mail-Adresse einer `User`-Entity im Backstage Catalog.


## 5. Benutzer im Backstage Catalog erfassen

Eine passende Benutzerdefinition erstellen, beispielsweise unter:

```text
examples/users.yaml
```


In [ ]:
apiVersion: backstage.io/v1alpha1
kind: User
metadata:
  name: marcel
spec:
  profile:
    displayName: Marcel
    email: deine-google-adresse@gmail.com
  memberOf: []

Die E-Mail-Adresse muss exakt der Adresse des verwendeten Google-Accounts entsprechen.

Die Datei wird anschliessend in `app-config.yaml` registriert:


In [ ]:
catalog:
  locations:
    - type: file
      target: ../../examples/users.yaml

Ohne passende `User`-Entity kann Backstage den angemeldeten Google-Benutzer keiner Catalog-Identität zuordnen.

Eine typische Fehlermeldung lautet:

```text
Failed to sign-in, unable to resolve user identity
```


## 6. Google-Login im Frontend anzeigen

Bei einer aktuellen Backstage-App mit dem neuen Frontend-System wird die Sign-in-Seite in:

```text
packages/app/src/App.tsx
```

konfiguriert.


In [ ]:
import { googleAuthApiRef } from '@backstage/core-plugin-api';
import { SignInPage } from '@backstage/core-components';
import { SignInPageBlueprint } from '@backstage/plugin-app-react';
import {
  createFrontendModule,
} from '@backstage/frontend-plugin-api';

const signInPage = SignInPageBlueprint.make({
  params: {
    loader: async () => props => (
      <SignInPage
        {...props}
        provider={{
          id: 'google-auth-provider',
          title: 'Google',
          message: 'Mit Google anmelden',
          apiRef: googleAuthApiRef,
        }}
      />
    ),
  },
});

Das Frontend-Modul wird danach in der bestehenden `createApp`-Konfiguration ergänzt:


In [ ]:
export default createApp({
  features: [
    // Bestehende Features

    createFrontendModule({
      pluginId: 'app',
      extensions: [signInPage],
    }),
  ],
});

## Ergebnis bis Punkt 6

Nach diesen Schritten sind folgende Bestandteile konfiguriert:

1. Google OAuth-Client
2. Google-Provider im Backstage-Backend
3. OAuth-Zugangsdaten über Umgebungsvariablen
4. Sign-in-Resolver für Catalog-Benutzer
5. Passende Backstage-User-Entity
6. Google-Login-Seite im Frontend

Zum Starten von Backstage kann anschliessend im Root-Verzeichnis ausgeführt werden:

```bash
yarn dev
```

Die Anwendung ist danach unter `http://localhost:3000` erreichbar.
